# Imputing wealth from the SCF to the CPS

This notebook demonstrates how to use the `microimpute` package and specifically the `autoimpute` function to impute wealth variables from the Survey of Consumer Finances to the Current Population Survey.

The Survey of Consumer Finances (SCF) is a triennial survey conducted by the Federal Reserve that collects detailed information on U.S. families' balance sheets, income, and demographic characteristics, with a special focus on wealth measures. The Current Population Survey (CPS) is a monthly survey conducted by the Census Bureau that provides comprehensive data on the labor force, employment, unemployment, and demographic characteristics, but lacks detailed wealth information.

By using `microimpute`, wealth information can be transfered from the SCF to the CPS, enabling economic analyses that require both detailed labor market and wealth data.

In [1]:
import io
import logging
import zipfile
import os
import subprocess

from typing import List, Optional, Type, Union

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pydantic import validate_call
from tqdm import tqdm
import warnings
import policyengine_us

from microimpute.config import VALIDATE_CONFIG, VALID_YEARS, PLOT_CONFIG
from microimpute.comparisons import *
from microimpute.visualizations import *
from microimpute.evaluations import *
from microimpute.utils.data import preprocess_data

logger = logging.getLogger(__name__)

## Loading and preparing the SCF and CPS datasets

The first step in the imputation process involves acquiring and harmonizing the two datasets. Extracting data from the SCF and the CPS, and then processing it to ensure the variables are compatible for imputation are crucial pre-processing steps for successful imputation. This involves identifying predictor variables that exist in both data sets and can meaningfully predict wealth, as well as ensuring they are named and encoded identically. 

In [2]:
@validate_call(config=VALIDATE_CONFIG)
def scf_url(year: int, VALID_YEARS: List[int] = VALID_YEARS) -> str:
    """Return the URL of the SCF summary microdata zip file for a year.

    Args:
        year: Year of SCF summary microdata to retrieve.

    Returns:
        URL of summary microdata zip file for the given year.

    Raises:
        ValueError: If the year is not in VALID_YEARS.
    """
    logger.debug(f"Generating SCF URL for year {year}")

    if year not in VALID_YEARS:
        logger.error(f"Invalid SCF year: {year}. Valid years are {VALID_YEARS}")
        raise ValueError(
            f"The SCF is not available for {year}. Valid years are {VALID_YEARS}"
        )

    url = f"https://www.federalreserve.gov/econres/files/scfp{year}s.zip"
    logger.debug(f"Generated URL: {url}")
    return url


@validate_call(config=VALIDATE_CONFIG)
def load_scf(
    years: Optional[Union[int, List[int]]] = VALID_YEARS,
    columns: Optional[List[str]] = None,
) -> pd.DataFrame:
    """Load Survey of Consumer Finances data for specified years and columns.

    Args:
        years: Year or list of years to load data for.
        columns: List of column names to load.

    Returns:
        DataFrame containing the requested data.

    Raises:
        ValueError: If no Stata files are found in the downloaded zip
            or invalid parameters
        RuntimeError: If there's a network error or a problem processing
            the downloaded data
    """

    logger.info(f"Loading SCF data with years={years}")

    try:
        # Identify years for download
        if years is None:
            years = VALID_YEARS
            logger.warning(f"Using default years: {years}")

        if isinstance(years, int):
            years = [years]

        # Validate all years are valid
        invalid_years = [year for year in years if year not in VALID_YEARS]
        if invalid_years:
            logger.error(f"Invalid years specified: {invalid_years}")
            raise ValueError(
                f"Invalid years: {invalid_years}. Valid years are {VALID_YEARS}"
            )

        all_data: List[pd.DataFrame] = []

        for year in tqdm(years):
            logger.info(f"Processing data for year {year}")
            try:
                # Download zip file
                logger.debug(f"Downloading SCF data for year {year}")
                url = scf_url(year)
                try:
                    response = requests.get(url, timeout=60)
                    response.raise_for_status()  # Raise an error for bad responses
                except requests.exceptions.RequestException as e:
                    logger.error(
                        f"Network error downloading SCF data for year {year}: {str(e)}"
                    )
                    raise RuntimeError(
                        f"Failed to download SCF data for year {year}"
                    ) from e

                # Process zip file
                try:
                    logger.debug("Creating zipfile from downloaded content")
                    z = zipfile.ZipFile(io.BytesIO(response.content))

                    # Find the .dta file in the zip
                    dta_files: List[str] = [
                        f for f in z.namelist() if f.endswith(".dta")
                    ]
                    if not dta_files:
                        logger.error(f"No Stata files found in zip for year {year}")
                        raise ValueError(f"No Stata files found in zip for year {year}")

                    logger.debug(f"Found Stata files: {dta_files}")

                    # Read the Stata file
                    try:
                        logger.debug(f"Reading Stata file: {dta_files[0]}")
                        with z.open(dta_files[0]) as f:
                            df = pd.read_stata(io.BytesIO(f.read()), columns=columns)
                            logger.debug(f"Read DataFrame with shape {df.shape}")

                        # Ensure 'wgt' is included
                        if (
                            columns is not None
                            and "wgt" not in df.columns
                            and "wgt" not in columns
                        ):
                            logger.debug("Re-reading with 'wgt' column added")
                            # Re-read to include weights
                            with z.open(dta_files[0]) as f:
                                cols_with_weight: List[str] = list(
                                    set(columns) | {"wgt"}
                                )
                                df = pd.read_stata(
                                    io.BytesIO(f.read()),
                                    columns=cols_with_weight,
                                )
                                logger.debug(f"Re-read DataFrame with shape {df.shape}")
                    except Exception as e:
                        logger.error(
                            f"Error reading Stata file for year {year}: {str(e)}"
                        )
                        raise RuntimeError(
                            f"Failed to process Stata file for year {year}"
                        ) from e

                except zipfile.BadZipFile as e:
                    logger.error(f"Bad zip file for year {year}: {str(e)}")
                    raise RuntimeError(
                        f"Downloaded zip file is corrupt for year {year}"
                    ) from e

                # Add year column
                df["year"] = year
                logger.info(
                    f"Successfully processed data for year {year}, shape: {df.shape}"
                )
                all_data.append(df)

            except Exception as e:
                logger.error(f"Error processing year {year}: {str(e)}")
                raise

        # Combine all years
        logger.debug(f"Combining data from {len(all_data)} years")
        if len(all_data) > 1:
            result = pd.concat(all_data)
            logger.info(
                f"Combined data from {len(years)} years, final shape: {result.shape}"
            )
            return result
        else:
            logger.info(f"Returning data for single year, shape: {all_data[0].shape}")
            return all_data[0]

    except Exception as e:
        logger.error(f"Error in _load: {str(e)}")
        raise


scf = load_scf(2022)

# Create mapping from desired variable names to SCF column names
scf_variable_mapping = {
    "hhsex": "is_female",  # sex (is female yes/no) (hhsex)
    "age": "age",  # age of respondent (age)
    "race": "race",  # race of respondent (race)
    "kids": "own_children_in_household",  # number of children in household (kids)
    "wageinc": "employment_income",  # income from wages and salaries (wageinc)
    "bussefarminc": "farm_self_employment_income",  # income from business, self-employment or farm (bussefarminc)
    "intdivinc": "interest_dividend_income",  # income from interest and dividends (intdivinc)
    "ssretinc": "pension_income",  # income from social security and retirement accounts (ssretinc)
}

original_columns = list(scf_variable_mapping.keys()) + ["networth", "wgt"]
scf_df = pd.DataFrame({col: scf[col] for col in original_columns})
scf_data = scf_df.rename(columns=scf_variable_mapping)

# Convert hhsex to is_female (hhsex: 1=male, 2=female -> is_female: 0=male, 1=female)
scf_data["is_female"] = (scf_data["is_female"] == 2).astype(int)

predictors = [
    "is_female",
    "age",
    "own_children_in_household",
    "race",
    "employment_income",
    "interest_dividend_income",
    "pension_income",
]

imputed_variables = ["networth"]

weights = ["wgt"]

scf_data = scf_data[predictors + imputed_variables + weights]

# Uprate SCF 2022 to 2024 to match EnhancedCPS_2024
from policyengine_core.periods import instant
from policyengine_us.system import CountryTaxBenefitSystem

_tax_benefit_system = CountryTaxBenefitSystem()
_params = _tax_benefit_system.parameters
cpi_2022 = float(_params.gov.bls.cpi.c_cpi_u(instant("2022-09-01")))
cpi_2024 = float(_params.gov.bls.cpi.c_cpi_u(instant("2024-09-01")))
cpi_uprating_factor = cpi_2024 / cpi_2022
print(
    f"CPI uprating factor (2022 -> 2024): "
    f"{cpi_uprating_factor:.4f} ({(cpi_uprating_factor - 1) * 100:.1f}%)"
)

# Uprate monetary variables (predictors + target)
monetary_cols = [
    "employment_income",
    "interest_dividend_income",
    "pension_income",
    "networth",
]
for col in monetary_cols:
    scf_data[col] = scf_data[col] * cpi_uprating_factor

# Age forward by 2 years (SCF 2022 -> 2024)
scf_data["age"] = scf_data["age"] + 2

100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


CPI uprating factor (2022 -> 2024): 1.0558 (5.6%)


In [3]:
import ssl
import requests
import h5py
from pathlib import Path

# Disable SSL verification warnings (only use in development environments)
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Create unverified context for SSL connections
ssl._create_default_https_context = ssl._create_unverified_context

# Monkey patch the requests library to use the unverified context
old_get = requests.get
requests.get = lambda *args, **kwargs: old_get(*args, **{**kwargs, "verify": False})

from policyengine_us_data import EnhancedCPS_2024

ENHANCED_CPS_PATH = str(EnhancedCPS_2024().file_path)


def load_dataset(file_path):
    """Load all datasets from H5 file into a flat dictionary.

    Handles both flat format (variable -> array) and nested
    time-period format (variable/year -> array) used by
    EnhancedCPS datasets.
    """
    with h5py.File(file_path, "r") as f:
        data = {}

        for table_name in f.keys():
            table = f[table_name]

            if isinstance(table, h5py.Group):
                for var_name in table.keys():
                    if isinstance(table[var_name], h5py.Dataset):
                        # Use the group name (variable name) as key,
                        # not the sub-key (year)
                        data[table_name] = table[var_name][:]

            elif isinstance(table, h5py.Dataset):
                data[table_name] = table[:]

        return data


cps = load_dataset(ENHANCED_CPS_PATH)

# Drop existing net_worth — the EnhancedCPS already contains
# QRF-imputed values; we remove them so our own imputation
# comparison starts from a clean slate.
cps.pop("net_worth", None)

# In the EnhancedCPS, employment_income and self_employment_income
# are computed variables (empty groups). The actual survey data is
# stored in the _before_lsr (before labor supply response) variants.
# Without a reform, the behavioral response is 0, so these are
# equivalent to the standard employment_income values.
cps["employment_income"] = cps["employment_income_before_lsr"]
cps["self_employment_income"] = cps["self_employment_income_before_lsr"]

cps_race_mapping = {
    1: 1,  # White only -> WHITE
    2: 2,  # Black only -> BLACK/AFRICAN-AMERICAN
    3: 5,  # American Indian, Alaskan Native only -> OTHER
    4: 4,  # Asian only -> ASIAN
    5: 5,  # Hawaiian/Pacific Islander only -> OTHER
    6: 5,  # White-Black -> OTHER
    7: 5,  # White-AI -> OTHER
    8: 5,  # White-Asian -> OTHER
    9: 3,  # White-HP -> HISPANIC
    10: 5,  # Black-AI -> OTHER
    11: 5,  # Black-Asian -> OTHER
    12: 3,  # Black-HP -> HISPANIC
    13: 5,  # AI-Asian -> OTHER
    14: 5,  # AI-HP -> OTHER
    15: 3,  # Asian-HP -> HISPANIC
    16: 5,  # White-Black-AI -> OTHER
    17: 5,  # White-Black-Asian -> OTHER
    18: 5,  # White-Black-HP -> OTHER
    19: 5,  # White-AI-Asian -> OTHER
    20: 5,  # White-AI-HP -> OTHER
    21: 5,  # White-Asian-HP -> OTHER
    22: 5,  # Black-AI-Asian -> OTHER
    23: 5,  # White-Black-AI-Asian -> OTHER
    24: 5,  # White-AI-Asian-HP -> OTHER
    25: 5,  # Other 3 race comb. -> OTHER
    26: 5,  # Other 4 or 5 race comb. -> OTHER
}

# Apply the mapping to recode the race values
cps["race"] = np.vectorize(cps_race_mapping.get)(cps["cps_race"])
cps["farm_self_employment_income"] = cps["self_employment_income"] + cps["farm_income"]
cps["interest_dividend_income"] = (
    cps["taxable_interest_income"]
    + cps["tax_exempt_interest_income"]
    + cps["qualified_dividend_income"]
    + cps["non_qualified_dividend_income"]
)
cps["pension_income"] = (
    cps["tax_exempt_private_pension_income"]
    + cps["taxable_private_pension_income"]
    + cps["social_security_retirement"]
)

mask_head = cps["is_household_head"]
income_df = pd.DataFrame(
    {
        "household_id": cps["person_household_id"],
        "employment_income": cps["employment_income"],
        "farm_self_employment_income": cps["farm_self_employment_income"],
        "interest_dividend_income": cps["interest_dividend_income"],
        "pension_income": cps["pension_income"],
    }
)
household_sums = income_df.groupby("household_id").sum().reset_index()
heads = pd.DataFrame(
    {
        "household_id": cps["person_household_id"][mask_head],
        "is_female": cps["is_female"][mask_head],
        "age": cps["age"][mask_head],
        "race": cps["race"][mask_head],
        "own_children_in_household": cps["own_children_in_household"][mask_head],
    }
)
hh_level = heads.merge(household_sums, on="household_id", how="left")

for name, series in cps.items():
    if isinstance(series, pd.Series) and len(series) == len(hh_level):
        if name not in hh_level.columns:
            hh_level[name] = series.values


cols = (
    ["household_id"]
    + [
        "farm_self_employment_income",
        "interest_dividend_income",
        "pension_income",
        "employment_income",
    ]
    + ["own_children_in_household", "is_female", "age", "race"]
)
cps_data = hh_level[cols]
cps_data["household_weight"] = cps["household_weight"]

household_weights = ["household_weight"]

In [4]:
from policyengine_us import Microsimulation
from policyengine_us_data import EnhancedCPS_2024

sim = Microsimulation(dataset=EnhancedCPS_2024)
net_disposable_income = sim.calculate("household_net_income", period=2024)

In [5]:
cps_data["household_net_income"] = net_disposable_income

In [6]:
weights_col = scf_data["wgt"].values
weights_normalized = weights_col / weights_col.sum()
scf_data_weighted = scf_data.sample(
    n=len(scf_data),
    replace=True,
    weights=weights_normalized,
).reset_index(drop=True)


# Create a combined plot comparing weighted and unweighted SCF distributions
def safe_log10(x):
    """Apply log10 to absolute values, preserving sign."""
    sign = np.sign(x)
    log_x = np.log10(np.maximum(np.abs(x), 1e-10))
    return sign * log_x


# Apply safe log transformation to both datasets
scf_log = safe_log10(scf_data["networth"])
scf_weighted_log = safe_log10(scf_data_weighted["networth"])

colors = PLOT_CONFIG["color_palette"]

# Create the figure
fig = go.Figure()

# Add unweighted SCF histogram
fig.add_trace(
    go.Histogram(
        x=scf_log,
        nbinsx=200,
        opacity=0.7,
        name="SCF Unweighted",
        marker_color=colors[0],
        histnorm="percent",
    )
)

# Add weighted SCF histogram
fig.add_trace(
    go.Histogram(
        x=scf_weighted_log,
        nbinsx=200,
        opacity=0.7,
        name="SCF Weighted (through sampling)",
        marker_color=colors[1],
        histnorm="percent",
    )
)

# Calculate statistics for both distributions
unweighted_median = np.median(scf_log)
weighted_median = np.median(scf_weighted_log)
unweighted_mean = np.mean(scf_log)
weighted_mean = np.mean(scf_weighted_log)

# Add median lines
fig.add_trace(
    go.Scatter(
        x=[unweighted_median, unweighted_median],
        y=[0, 8],
        mode="lines",
        line=dict(color=colors[0], width=2, dash="dash"),
        name=f"Unweighted Median: ${10**unweighted_median:,.0f}",
    )
)

fig.add_trace(
    go.Scatter(
        x=[weighted_median, weighted_median],
        y=[0, 8],
        mode="lines",
        line=dict(color=colors[1], width=2, dash="dash"),
        name=f"Weighted Median: ${10**weighted_median:,.0f}",
    )
)

# Update layout
fig.update_layout(
    title="SCF net worth distribution: weighted vs unweighted (log scale)",
    width=PLOT_CONFIG["width"],
    height=PLOT_CONFIG["height"],
    barmode="overlay",
    bargap=0.1,
    paper_bgcolor=PLOT_CONFIG["paper_bgcolor"],
    plot_bgcolor=PLOT_CONFIG["plot_bgcolor"],
    legend=dict(
        x=0.01,
        y=0.99,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(0,0,0,0.3)",
        borderwidth=1,
        orientation="v",
        xanchor="left",
        yanchor="top",
    ),
)

# Custom tick formatting for x-axis
tick_values = [-6, -4, -2, 0, 2, 4, 6, 8, 10]
tick_labels = []
for x in tick_values:
    if x >= 0:
        tick_labels.append(f"${10**x:,.0f}")
    else:
        tick_labels.append(f"-${10 ** abs(x):,.0f}")

fig.update_xaxes(
    tickvals=tick_values,
    ticktext=tick_labels,
    title_text="Net worth (log10 scale)",
    showgrid=PLOT_CONFIG["showgrid_x"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(
    title_text="Percentage",
    showgrid=PLOT_CONFIG["showgrid_y"],
    gridcolor=PLOT_CONFIG["gridcolor"],
    gridwidth=PLOT_CONFIG["gridwidth"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)

# Add annotations with statistics
fig.add_annotation(
    x=0.98,
    y=0.95,
    xref="paper",
    yref="paper",
    text=(
        f"<b>Unweighted:</b><br>"
        f"Median: ${10**unweighted_median:,.0f}<br>"
        f"Mean: ${10**unweighted_mean:,.0f}<br><br>"
        f"<b>Weighted:</b><br>"
        f"Median: ${10**weighted_median:,.0f}<br>"
        f"Mean: ${10**weighted_mean:,.0f}"
    ),
    showarrow=False,
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="rgba(0,0,0,0.3)",
    borderwidth=1,
    font=dict(size=11),
    xanchor="right",
    yanchor="top",
)

fig.show()

The logarithmic transformation provides a clearer view of the wealth distribution across its entire range, as it compresses values that span many orders of magnitude. By logarithmically scaling the data, the extreme values are compressed while expanding the visibility of differences in the lower and middle portions of the distribution. Additionally, by using sampling weights for visualizing the distribution and adjusting the training data, we ensure that models learn the true wealth distribution, rather than the right-skewed representation that the SCF dataset produces with its sampling design.

## Running wealth imputation with autoimpute

After harmonizing the two datasets, the `autoimpute` function from `microimpute` can be used to transfer wealth information from the SCF to the CPS. This powerful function streamlines the imputation process by automating hyperparameter tuning, method selection, validation, and application.

Behind the scenes, `autoimpute` evaluates multiple statistical approaches, including Quantile Regression Forest, Ordinary Least Squares, Quantile Regression, and Statistical Matching. It performs cross-validation to determine which method most accurately captures the relationship between the predictor variables and wealth measures in the SCF data. The function then applies the best-performing method to generate synthetic wealth values for CPS households.

By enabling hyperparameter tuning, the function can optimize each method's parameters, further improving imputation accuracy. This automated approach saves considerable time and effort compared to manually testing different imputation strategies, while ensuring the selection of the most appropriate method for this specific imputation task.

In [7]:
warnings.filterwarnings("ignore")

# Run the autoimpute process
autoimpute_results = autoimpute(
    donor_data=scf_data,
    receiver_data=cps_data,
    predictors=predictors,
    imputed_variables=imputed_variables,
    weight_col="wgt",
    tune_hyperparameters=True,  # enable automated hyperparameter tuning
    impute_all=True,
    preprocessing={
        "networth": "asinh",
    },
    force_retrain=True,
)

logger.info(
    f"Shape of receiver data before imputation: {cps_data.shape} \nShape of receiver data after imputation: {autoimpute_results.receiver_data.shape}"
)

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  4.2min remaining:  6.2min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  4.2min remaining:  2.8min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  4.2min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.9s remaining:    1.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.9s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    1.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   31.0s remaining:   46.4s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   33.3s remaining:   22.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   34.2s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed: 10.9min
[Parallel(n_

## Comparing method performance

The method comparison plot below shows how different imputation methods performed across various quantiles. Lower quantile loss values indicate better performance. 

In [8]:
from microimpute.visualizations.comparison_plots import method_comparison_results

comparison_viz = method_comparison_results(
    data=autoimpute_results.cv_results,
    metric="quantile_loss",
    data_format="wide",
)
fig = comparison_viz.plot(
    title="Autoimpute method comparison",
    show_mean=True,
)
fig.show()

## Evaluating predictors' influence on the imputation results

To understand the relationship between the variables we have selected as predictors and the imputation results obtained, we can assess correlation between predictors, and the sensitivity that our results demonstrate to a different number or sequence of predictors used.

In [9]:
from microimpute.evaluations.predictor_analysis import compute_predictor_correlations

correlations = compute_predictor_correlations(scf_data, predictors, imputed_variables)

In [10]:
correlations["pearson"]

,is_female,age,own_children_in_household,race,employment_income,interest_dividend_income,pension_income
is_female,1.000000,-0.018881,-0.063568,0.057994,-0.085917,-0.037146,-0.055694
age,-0.018881,1.000000,-0.327560,-0.204776,0.018204,0.093807,0.283672
own_children_in_household,-0.063568,-0.327560,1.000000,0.155980,0.013922,-0.040177,-0.130564
race,0.057994,-0.204776,0.155980,1.000000,-0.046325,-0.036117,-0.111804
employment_income,-0.085917,0.018204,0.013922,-0.046325,1.000000,0.056797,0.040700
interest_dividend_income,-0.037146,0.093807,-0.040177,-0.036117,0.056797,1.000000,0.121772
pension_income,-0.055694,0.283672,-0.130564,-0.111804,0.040700,0.121772,1.000000


In [11]:
correlations["predictor_target_mi"]

,networth
is_female,0.014095
age,0.119559
own_children_in_household,0.022649
race,0.027996
employment_income,0.121915
interest_dividend_income,0.085286
pension_income,0.059902


In [12]:
from microimpute.evaluations.predictor_analysis import (
    progressive_predictor_inclusion,
    leave_one_out_analysis,
)
from microimpute.models import QRF

leave_one_out_results = leave_one_out_analysis(
    scf_data, predictors, imputed_variables, QRF
)

predictor_inclusion_results = progressive_predictor_inclusion(
    scf_data, predictors, imputed_variables, QRF
)

Leave-one-out analysis:   0%|          | 0/7 [00:00<?, ?it/s]

Progressive inclusion:   0%|          | 0/7 [00:00<?, ?it/s]

In [13]:
leave_one_out_results

,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
5,interest_dividend_income,3.240984e+06,0,1.668592e+06,106.118060,1.572392e+06,0
4,employment_income,2.011281e+06,0,4.388883e+05,27.912140,1.572392e+06,0
1,age,1.948841e+06,0,3.764485e+05,23.941130,1.572392e+06,0
6,pension_income,1.691464e+06,0,1.190717e+05,7.572646,1.572392e+06,0
2,own_children_in_household,1.661723e+06,0,8.933042e+04,5.681179,1.572392e+06,0
3,race,1.614550e+06,0,4.215818e+04,2.681149,1.572392e+06,0
0,is_female,1.584438e+06,0,1.204559e+04,0.766068,1.572392e+06,0


In [14]:
print(f"Optimal subset: {predictor_inclusion_results['optimal_subset']}")
print(f"Optimal loss: {predictor_inclusion_results['optimal_loss']}")

# For step-by-step details:
print(predictor_inclusion_results["results_df"])

Optimal subset: ['interest_dividend_income', 'age', 'employment_income', 'pension_income', 'own_children_in_household', 'race', 'is_female']
Optimal loss: 1568962.659746099
   step            predictor_added  \
0     1   interest_dividend_income   
1     2                        age   
2     3          employment_income   
3     4             pension_income   
4     5  own_children_in_household   
5     6                       race   
6     7                  is_female   

                                 predictors_included  avg_quantile_loss  \
0                         [interest_dividend_income]       5.037628e+06   
1                    [interest_dividend_income, age]       2.752098e+06   
2  [interest_dividend_income, age, employment_inc...       1.932222e+06   
3  [interest_dividend_income, age, employment_inc...       1.737197e+06   
4  [interest_dividend_income, age, employment_inc...       1.632692e+06   
5  [interest_dividend_income, age, employment_inc...       1.584996e+06 

## Evaluating wealth imputations

To assess the imputation results, a comparison of the distribution of wealth in the original SCF data with the imputed values in the CPS allows examining how well the imputation preserves important characteristics of the wealth distribution, such as its shape, central tendency, and dispersion.

Wealth distributions are typically highly skewed, with a long right tail representing a small number of households with very high net worth. A successful imputation should preserve this characteristic skewness while maintaining realistic values across the entire distribution. Examining both the raw distributions and log-transformed versions of wealth values can better capture important information for evaluation.

In [15]:
from microimpute.comparisons.metrics import compare_distributions

for model, imputations in autoimpute_results.imputations.items():
    print(
        f"Model: {model}, distribution similarity: \n{
            compare_distributions(
                donor_data=pd.DataFrame(scf_data['networth']),
                receiver_data=pd.DataFrame(imputations['networth']),
                donor_weights=scf_data['wgt'],
                receiver_weights=cps_data['household_weight'],
                imputed_variables=imputed_variables,
            )
        }"
    )
    if model == "best_method":
        distribution_comparison_results = compare_distributions(
            donor_data=pd.DataFrame(scf_data["networth"]),
            receiver_data=pd.DataFrame(imputations["networth"]),
            donor_weights=scf_data["wgt"],
            receiver_weights=cps_data["household_weight"],
            imputed_variables=imputed_variables,
        )

Model: best_method, distribution similarity: 
   Variable                Metric       Distance
0  networth  wasserstein_distance  910298.510146
Model: OLS, distribution similarity: 
   Variable                Metric      Distance
0  networth  wasserstein_distance  6.799487e+07
Model: QuantReg, distribution similarity: 
   Variable                Metric      Distance
0  networth  wasserstein_distance  1.217437e+14
Model: Matching, distribution similarity: 
   Variable                Metric      Distance
0  networth  wasserstein_distance  1.189243e+06
Model: MDN, distribution similarity: 
   Variable                Metric      Distance
0  networth  wasserstein_distance  2.639191e+14


In [29]:
# Robustness check: trimmed Wasserstein distances (excluding top/bottom 1%)
# This verifies that the method ranking is not driven by extreme-tail leverage.

p1 = np.percentile(scf_data["networth"], 1)
p99 = np.percentile(scf_data["networth"], 99)

donor_mask = (scf_data["networth"] >= p1) & (scf_data["networth"] <= p99)
donor_dropped = len(scf_data) - donor_mask.sum()

print(f"Trimming to [{p1:,.0f}, {p99:,.0f}] (1st-99th percentile of SCF net worth)")
print(
    f"Donor records: {len(scf_data):,} total, {donor_dropped:,} dropped ({donor_dropped / len(scf_data) * 100:.1f}%)\n"
)

for model, imputations in autoimpute_results.imputations.items():
    receiver_mask = (imputations["networth"] >= p1) & (imputations["networth"] <= p99)
    receiver_dropped = len(imputations) - receiver_mask.sum()

    print(
        f"Model: {model}, "
        f"receiver dropped: {receiver_dropped:,} of {len(imputations):,} ({receiver_dropped / len(imputations) * 100:.1f}%)"
    )
    print(
        f"  trimmed distribution similarity: \n{
            compare_distributions(
                donor_data=pd.DataFrame(scf_data['networth'][donor_mask]),
                receiver_data=pd.DataFrame(imputations['networth'][receiver_mask]),
                donor_weights=scf_data['wgt'][donor_mask],
                receiver_weights=cps_data['household_weight'][receiver_mask],
                imputed_variables=imputed_variables,
            )
        }"
    )

Trimming to [-76,757, 494,947,126] (1st-99th percentile of SCF net worth)
Donor records: 22,975 total, 459 dropped (2.0%)

Model: best_method, receiver dropped: 227 of 18,395 (1.2%)
  trimmed distribution similarity: 
   Variable                Metric       Distance
0  networth  wasserstein_distance  892190.030085
Model: OLS, receiver dropped: 23 of 18,395 (0.1%)
  trimmed distribution similarity: 
   Variable                Metric       Distance
0  networth  wasserstein_distance  742528.269654
Model: QuantReg, receiver dropped: 76 of 18,395 (0.4%)
  trimmed distribution similarity: 
   Variable                Metric       Distance
0  networth  wasserstein_distance  650018.336012
Model: Matching, receiver dropped: 306 of 18,395 (1.7%)
  trimmed distribution similarity: 
   Variable                Metric      Distance
0  networth  wasserstein_distance  1.209626e+06
Model: MDN, receiver dropped: 61 of 18,395 (0.3%)
  trimmed distribution similarity: 
   Variable                Metric    

In [ ]:
def plot_all_models_log_distributions(
    donor_data,
    model_results,
    donor_weights,
    receiver_weights,
    title=None,
):
    """Plot log-transformed net worth distributions in a 3x2 grid.

    First subplot shows only the weighted donor distribution.
    Remaining five subplots overlay each model's imputations with the donor.

    Args:
        donor_data: Original donor data with networth column
        model_results: Dictionary mapping model names to their imputed dataframes
            (should contain exactly 5 models)
        donor_weights: Weights for the donor data
        receiver_weights: Weights for the receiver data
        title: Optional title for the entire figure

    Returns:
        Plotly figure with 6 subplots (percentage histograms)
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go

    colors = [
        "#CC6677",  # Rose
        "#DDCC77",  # Sand
        "#117733",  # Green
        "#332288",  # Indigo
        "#88CCEE",  # Cyan
    ]

    # Normalize weights
    donor_weights_normalized = donor_weights / donor_weights.sum()
    receiver_weights_normalized = receiver_weights / receiver_weights.sum()

    # Sample donor data with weights
    donor_sampled = donor_data.sample(
        n=len(donor_data),
        replace=True,
        weights=donor_weights_normalized,
        random_state=405,
    ).reset_index(drop=True)

    # Define safe log transformation
    def safe_log(x):
        sign = np.sign(x)
        log_x = np.log10(np.maximum(np.abs(x), 1e-10))
        return sign * log_x

    # Calculate donor log values
    donor_log = safe_log(donor_sampled["networth"])
    donor_log_median = np.median(donor_log)
    donor_log_mean = np.mean(donor_log)

    # Subplot titles: first is donor only, rest are models
    model_names = list(model_results.keys())
    subplot_titles = ["SCF donor distribution (weighted)"] + model_names

    # Create 3x2 subplots (3 rows, 2 columns)
    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.10,
        horizontal_spacing=0.08,
    )

    # Define colors
    donor_color = "#999999"

    # Custom tick formatting
    tick_values = [-6, -4, -2, 0, 2, 4, 6, 8]
    tick_labels = [
        "$" + format(10**x if x >= 0 else -(10 ** abs(x)), ",.0f") for x in tick_values
    ]

    # Plot positions for 3x2 grid (row, col)
    positions = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 1), (3, 2)]

    # First subplot: Donor distribution only
    row, col = positions[0]
    fig.add_trace(
        go.Histogram(
            x=donor_log,
            nbinsx=200,
            opacity=0.7,
            name="SCF (weighted)",
            marker_color=donor_color,
            histnorm="percent",
            showlegend=True,
        ),
        row=row,
        col=col,
    )

    # Add median line for donor
    fig.add_trace(
        go.Scatter(
            x=[donor_log_median, donor_log_median],
            y=[0, 12],
            mode="lines",
            line=dict(color=donor_color, width=2, dash="dash"),
            name=f"SCF Median: ${10**donor_log_median:,.0f}",
            showlegend=True,
        ),
        row=row,
        col=col,
    )

    # Add statistics annotation for donor subplot
    fig.add_annotation(
        x=7,
        y=9,
        xref="x",
        yref="y",
        text=f"Median: ${10**donor_log_median:,.0f}<br>Mean: ${10**donor_log_mean:,.0f}",
        showarrow=False,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(0,0,0,0.3)",
        borderwidth=1,
        font=dict(size=10),
        xanchor="right",
        yanchor="top",
    )

    # Remaining subplots: Model imputations overlaid with donor
    for idx, (model_name, imputed_data) in enumerate(model_results.items()):
        subplot_idx = idx + 1  # Skip first subplot
        row, col = positions[subplot_idx]
        model_color = colors[idx % len(colors)]

        # Sample receiver data with weights
        receiver_sampled = imputed_data.sample(
            n=len(imputed_data),
            replace=True,
            weights=receiver_weights_normalized,
            random_state=405,
        ).reset_index(drop=True)

        # Calculate model log values
        model_log = safe_log(receiver_sampled["networth"])
        model_log_median = np.median(model_log)
        model_log_mean = np.mean(model_log)

        # Add donor histogram (grey/transparent background)
        fig.add_trace(
            go.Histogram(
                x=donor_log,
                nbinsx=200,
                opacity=0.5,
                name="SCF (weighted)",
                marker_color=donor_color,
                histnorm="percent",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Add model histogram
        fig.add_trace(
            go.Histogram(
                x=model_log,
                nbinsx=200,
                opacity=0.6,
                name=model_name.replace(" imputations", ""),
                marker_color=model_color,
                histnorm="percent",
                showlegend=True,
            ),
            row=row,
            col=col,
        )

        # Add donor median line (grey dashed)
        fig.add_trace(
            go.Scatter(
                x=[donor_log_median, donor_log_median],
                y=[0, 12],
                mode="lines",
                line=dict(color=donor_color, width=2, dash="dash"),
                name="SCF Median",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Add model median line
        fig.add_trace(
            go.Scatter(
                x=[model_log_median, model_log_median],
                y=[0, 12],
                mode="lines",
                line=dict(color=model_color, width=2, dash="dash"),
                name=f"{model_name} Median",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        # Determine axis references for annotations
        axis_num = subplot_idx + 1
        if axis_num == 1:
            xref, yref = "x", "y"
        else:
            xref, yref = f"x{axis_num}", f"y{axis_num}"

        # Add statistics annotation
        fig.add_annotation(
            x=7,
            y=9,
            xref=xref,
            yref=yref,
            text=f"Median: ${10**model_log_median:,.0f}<br>Mean: ${10**model_log_mean:,.0f}",
            showarrow=False,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="rgba(0,0,0,0.3)",
            borderwidth=1,
            font=dict(size=10),
            xanchor="right",
            yanchor="top",
        )

    # Update layout
    fig.update_layout(
        title=dict(
            text=(title if title else "Log-transformed net worth imputations by method"),
            font=dict(size=18),
            x=0.5,
            xanchor="center",
        ),
        height=900,
        width=1000,
        showlegend=True,
        paper_bgcolor=PLOT_CONFIG["paper_bgcolor"],
        plot_bgcolor=PLOT_CONFIG["plot_bgcolor"],
        legend=dict(
            x=0.5,
            y=-0.15,
            xanchor="center",
            yanchor="top",
            orientation="h",
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="rgba(0,0,0,0.3)",
            borderwidth=1,
        ),
        barmode="overlay",
        margin=dict(t=80, r=80, b=80),
    )

    # Update all axes
    for i in range(1, 7):
        row_num = (i - 1) // 2 + 1
        col_num = (i - 1) % 2 + 1

        fig.update_xaxes(
            tickvals=tick_values,
            ticktext=tick_labels,
            title_text="Net worth (log scale)" if row_num == 3 else "",
            showgrid=PLOT_CONFIG["showgrid_x"],
            showline=PLOT_CONFIG["showline"],
            linecolor=PLOT_CONFIG["linecolor"],
            row=row_num,
            col=col_num,
        )
        fig.update_yaxes(
            range=[0, 12],
            title_text="Percentage" if col_num == 1 else "",
            showgrid=PLOT_CONFIG["showgrid_y"],
            gridcolor=PLOT_CONFIG["gridcolor"],
            gridwidth=PLOT_CONFIG["gridwidth"],
            showline=PLOT_CONFIG["showline"],
            linecolor=PLOT_CONFIG["linecolor"],
            row=row_num,
            col=col_num,
        )

    return fig

In [ ]:
# Access tuned hyperparameters for each model
for model_name, fitted_model in autoimpute_results.fitted_models.items():
    print(f"\n{'=' * 50}")
    print(f"Model: {model_name}")
    print(f"{'=' * 50}")

    # QRF: params stored on the internal RandomForestQuantileRegressor
    if hasattr(fitted_model, "models"):
        for var_name, m in fitted_model.models.items():
            if hasattr(m, "qrf"):  # QRF numeric target
                print(f"  {var_name} (QRF):")
                print(f"    n_estimators:      {m.qrf.n_estimators}")
                print(f"    min_samples_split: {m.qrf.min_samples_split}")
                print(f"    min_samples_leaf:  {m.qrf.min_samples_leaf}")
                print(f"    max_features:      {m.qrf.max_features}")
                print(f"    bootstrap:         {m.qrf.bootstrap}")
            elif hasattr(m, "num_gaussian"):  # MDN
                print(f"  {var_name} (MDN):")
                print(f"    num_gaussian:   {m.num_gaussian}")
                print(f"    learning_rate:  {m.learning_rate}")

    # Matching: params stored in .hyperparameters dict
    if hasattr(fitted_model, "hyperparameters"):
        print(f"  Matching hyperparameters: {fitted_model.hyperparameters}")


Model: best_method
  networth (QRF):
    n_estimators:      202
    min_samples_split: 5
    min_samples_leaf:  1
    max_features:      0.953996983528
    bootstrap:         True

Model: OLS

Model: QuantReg

Model: Matching
  Matching hyperparameters: {'dist_fun': 'Mahalanobis', 'k': 4}

Model: MDN
  networth (MDN):
    num_gaussian:   5
    learning_rate:  0.001


In [28]:
# Create dictionary of model results
model_results = autoimpute_results.imputations.copy()
model_results["QRF"] = model_results["best_method"]
del model_results["best_method"]

# Create and show the combined plot
combined_fig = plot_all_models_log_distributions(
    scf_data,
    model_results,
    donor_weights=scf_data["wgt"],
    receiver_weights=cps_data["household_weight"],
)
combined_fig.show()

Comparing the wealth distributions that result from imputing from the SCF on to the CPS with five different models, we can visually recognize the different strengths and limitations of each of them. The implications of using one model instead of another for imputation will be further explored by evaluating the impact wealth imputed data has on microsimulation results.

## Wealth distributions by disposable income deciles

In [ ]:
income_col = "household_net_income"
wealth_col = "networth"

income_deciles = net_disposable_income.decile_rank()
decile_means = []

for model, imputations in model_results.items():
    tmp = cps_data.copy()
    cps_data[wealth_col] = imputations

    # Create a temporary dataframe with the imputed values
    tmp = pd.DataFrame(
        {
            "networth": imputations.values.flatten(),
            "income_decile": income_deciles.values,
        }
    )

    # Mean wealth in each decile
    out = (
        tmp.groupby("income_decile")["networth"]
        .median()
        .reset_index(name="median_wealth")
    )
    out["Method"] = model
    decile_means.append(out)

avg_df = pd.concat(decile_means, ignore_index=True)

fig = px.bar(
    avg_df,
    x="income_decile",
    y="median_wealth",
    color="Method",
    color_discrete_sequence=[
        "#CC6677",  # Rose
        "#DDCC77",  # Sand
        "#117733",  # Green
        "#332288",  # Indigo
        "#88CCEE",  # Cyan
    ],
    barmode="group",
    labels={
        "income_decile": "Net-income decile (1 = lowest, 10 = highest)",
        "median_wealth": "Median household net worth ($)",
    },
    title=(
        "Median household net worth by net-income decile<br>"
        "<sup>Comparison of imputation methods</sup>"
    ),
)

fig.update_layout(
    width=PLOT_CONFIG["width"],
    height=PLOT_CONFIG["height"],
    xaxis=dict(dtick=1, tick0=1),
    paper_bgcolor=PLOT_CONFIG["paper_bgcolor"],
    plot_bgcolor=PLOT_CONFIG["plot_bgcolor"],
    yaxis_tickformat="$,.0f",
    hovermode="x unified",
)

fig.update_xaxes(
    showgrid=PLOT_CONFIG["showgrid_x"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(
    showgrid=PLOT_CONFIG["showgrid_y"],
    gridcolor=PLOT_CONFIG["gridcolor"],
    gridwidth=PLOT_CONFIG["gridwidth"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)

fig.show()

## Loading results onto the microimputation dashboard

In [21]:
from microimpute.utils import format_csv

imputation_results_combined = format_csv(
    output_path="/Users/movil1/Desktop/PYTHONJOBS/PolicyEngine/microimpute/paper/scf_to_cps_imputation_results.csv",
    autoimpute_result=autoimpute_results.cv_results,
    comparison_metrics_df=None,
    distribution_comparison_df=distribution_comparison_results,
    predictor_correlations=correlations,
    predictor_importance_df=leave_one_out_results,
    progressive_inclusion_df=predictor_inclusion_results["results_df"],
    best_method_name="QRF",
    donor_data=scf_data,
    receiver_data=autoimpute_results.receiver_data,
    imputed_variables=imputed_variables,
    n_histogram_bins=150,
)

## SSI Policy Analysis: Baseline (No Wealth Data)

Supplemental Security Income (SSI) is a means-tested program that
requires applicants to have countable resources below $2,000
(individual) or $3,000 (couple). Since the CPS does not collect
wealth data, PolicyEngine's `ssi_countable_resources` variable
defaults to $0 for everyone — meaning all categorically eligible
persons (aged, blind, or disabled) pass the resource test. Then 
40% are randomly chosen as actually passing the test.

This baseline uses the Enhanced CPS 2024 calibrated dataset
whose survey weights have been reweighted to match administrative
totals for SSI participation, tax revenue, SNAP, and other programs.

Administrative totals come from the
[SSI Annual Statistical Report, 2024](https://www.ssa.gov/policy/docs/statcomps/ssi_asr/index.html).

In [ ]:
# SSI Baseline: No wealth data (all eligible pass resource test)

from policyengine_us import Microsimulation
from policyengine_us_data import EnhancedCPS_2024

sim_baseline = Microsimulation(dataset=EnhancedCPS_2024)

# PolicyEngine's .sum() already applies survey weights internally
ssi = sim_baseline.calculate("ssi", period=2024)

# Weighted totals via PolicyEngine's built-in weighted aggregation
total_expenditure = float(ssi.sum())
total_recipients = float((ssi > 0).astype(float).sum())
avg_monthly_benefit = (
    total_expenditure / total_recipients / 12 if total_recipients > 0 else 0
)

# Administrative totals (SSI Annual Statistical Report, 2024)
admin_recipients = 7_400_000
admin_expenditure = 59.6e9  # federal SSI payments, CY 2024
admin_avg_monthly = 697  # average monthly payment, Dec 2024

print("=" * 65)
print("SSI BASELINE: No wealth data (all eligible pass resource test)")
print("=" * 65)
print(f"{'Metric':<35} {'Simulated':>14} {'Admin Total':>14}")
print("-" * 65)
print(f"{'Total recipients':<35} {total_recipients:>14,.0f} {admin_recipients:>14,}")
print(
    f"{'Total expenditure ($B)':<35} "
    f"{total_expenditure / 1e9:>14.2f} "
    f"{admin_expenditure / 1e9:>14.1f}"
)
print(
    f"{'Avg monthly benefit ($)':<35} "
    f"{avg_monthly_benefit:>14,.0f} "
    f"{admin_avg_monthly:>14,}"
)
print("-" * 65)
print(
    f"{'Recipient ratio (sim/admin)':<35} {total_recipients / admin_recipients:>14.2%}"
)
print(
    f"{'Expenditure ratio (sim/admin)':<35} "
    f"{total_expenditure / admin_expenditure:>14.2%}"
)

SSI BASELINE: No wealth data (all eligible pass resource test)
Metric                                   Simulated    Admin Total
-----------------------------------------------------------------
Total recipients                        19,769,600      7,400,000
Total expenditure ($B)                      152.59           59.6
Avg monthly benefit ($)                        643            697
-----------------------------------------------------------------
Recipient ratio (sim/admin)                267.16%
Expenditure ratio (sim/admin)              256.02%


## SSI Policy Analysis: Impact of Imputed Wealth

With imputed net worth from each model, we can now test how the SSI
resource test changes outcomes. By default, PolicyEngine's
microsimulation assigns the resource test probabilistically (using a
40% pass rate parameter), because the CPS lacks wealth data. Imputation 
enables moving away from this, more accurately capturing which 
eligible persons would actually pass teh resource test based on 
their imputed net worth. We override the 40% random pass rate with a 
Reform that applies the actual resource-limit logic using our 5 different 
imputed net worth distributions.

Each model's household-level net worth is mapped to the person-level
`ssi_countable_resources` variable (assigned to household heads; the
`marital_unit.sum` in `meets_ssi_resource_test` handles joint claims).
Households with imputed net worth above the $2,000 individual /
$3,000 couple threshold will fail the resource test and lose SSI
eligibility. The magnitude of this effect depends on the wealth
distribution each imputation model produces.

In [ ]:
# SSI with imputed wealth as countable resources

from policyengine_us import Microsimulation
from policyengine_us.model_api import Variable, YEAR, USD, where
from policyengine_us.entities import Person
from policyengine_core.reforms import Reform
from policyengine_us_data import EnhancedCPS_2024

person_hh_ids = cps["person_household_id"]
is_head = cps["is_household_head"].astype(bool)
hh_ids = cps_data["household_id"].values


def make_ssi_resource_reform(resources_array):
    """Reform that injects imputed resources and bypasses the
    microsimulation probabilistic shortcut in meets_ssi_resource_test
    so that actual resource-limit logic is used instead."""
    arr = resources_array.copy()

    class ssi_countable_resources(Variable):
        value_type = float
        entity = Person
        label = "SSI countable resources"
        unit = USD
        definition_period = YEAR

        def formula(person, period, parameters):
            return arr

    class meets_ssi_resource_test(Variable):
        value_type = bool
        entity = Person
        label = "Meets SSI resource test"
        unit = USD
        definition_period = YEAR

        def formula(person, period, parameters):
            p = parameters(period).gov.ssa.ssi
            joint_claim = person("ssi_claim_is_joint", period)
            personal_resources = person("ssi_countable_resources", period)
            countable_resources = where(
                joint_claim,
                person.marital_unit.sum(personal_resources),
                personal_resources,
            )
            resource_limit = where(
                joint_claim,
                p.eligibility.resources.limit.couple,
                p.eligibility.resources.limit.individual,
            )
            return countable_resources <= resource_limit

    class reform(Reform):
        def apply(self):
            self.update_variable(ssi_countable_resources)
            self.update_variable(meets_ssi_resource_test)

    return reform


ssi_results = {}

for model_name, imputations in model_results.items():
    # Map household-level imputed net worth to person-level
    person_df = pd.DataFrame({"household_id": person_hh_ids})
    hh_df = pd.DataFrame(
        {
            "household_id": hh_ids,
            "networth": imputations["networth"].values.flatten(),
        }
    )
    merged = person_df.merge(hh_df, on="household_id", how="left")
    person_networth = merged["networth"].fillna(0).values

    # Assign to household heads only (non-heads get 0);
    # meets_ssi_resource_test uses marital_unit.sum for joint
    # claims, so the head's value propagates correctly.
    person_resources = np.where(is_head, np.maximum(person_networth, 0), 0.0).astype(
        np.float32
    )

    # Create simulation with reform that uses actual resource logic
    sim = Microsimulation(
        dataset=EnhancedCPS_2024,
        reform=make_ssi_resource_reform(person_resources),
    )
    ssi = sim.calculate("ssi", period=2024)

    total_exp = float(ssi.sum())
    total_recip = float((ssi > 0).astype(float).sum())
    avg_mo = total_exp / total_recip / 12 if total_recip > 0 else 0

    ssi_results[model_name] = {
        "recipients": total_recip,
        "expenditure": total_exp,
        "avg_monthly": avg_mo,
    }

# Add baseline (no wealth data) for comparison
ssi_results["Baseline (no wealth)"] = {
    "recipients": total_recipients,
    "expenditure": total_expenditure,
    "avg_monthly": avg_monthly_benefit,
}

# Comparison table 
print("=" * 80)
print("SSI OUTCOMES BY IMPUTATION MODEL")
print("(Imputed net worth used as ssi_countable_resources)")
print("=" * 80)
print(
    f"{'Model':<25} {'Recipients':>12} {'Expend ($B)':>12} "
    f"{'Avg Mo ($)':>11} {'Recip %':>9}"
)
print("-" * 80)
for model, r in ssi_results.items():
    print(
        f"{model:<25} "
        f"{r['recipients']:>12,.0f} "
        f"{r['expenditure'] / 1e9:>12.2f} "
        f"{r['avg_monthly']:>11,.0f} "
        f"{r['recipients'] / admin_recipients:>8.1%}"
    )
print("-" * 80)
print(
    f"{'Admin total':<25} "
    f"{admin_recipients:>12,} "
    f"{admin_expenditure / 1e9:>12.1f} "
    f"{admin_avg_monthly:>11,} "
    f"{'100.0%':>9}"
)

# Visualization
model_colors = [
    "#CC6677",  # OLS
    "#DDCC77",  # QuantReg
    "#117733",  # Matching
    "#332288",  # MDN
    "#88CCEE",  # QRF
    "#999999",  # Baseline
]

models = list(ssi_results.keys())
recipients = [r["recipients"] for r in ssi_results.values()]
expenditures = [r["expenditure"] / 1e9 for r in ssi_results.values()]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "SSI recipients by model",
        "SSI expenditure by model ($B)",
    ],
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Bar(
        x=models,
        y=recipients,
        marker_color=model_colors[: len(models)],
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_hline(
    y=admin_recipients,
    line_dash="dash",
    line_color="black",
    line_width=1,
    annotation_text="Admin total",
    annotation_position="top right",
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=models,
        y=expenditures,
        marker_color=model_colors[: len(models)],
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_hline(
    y=admin_expenditure / 1e9,
    line_dash="dash",
    line_color="black",
    line_width=1,
    annotation_text="Admin total",
    annotation_position="top right",
    row=1,
    col=2,
)

fig.update_layout(
    title="SSI outcomes with imputed wealth as countable resources",
    width=PLOT_CONFIG["width"] + 200,
    height=PLOT_CONFIG["height"] - 100,
    paper_bgcolor=PLOT_CONFIG["paper_bgcolor"],
    plot_bgcolor=PLOT_CONFIG["plot_bgcolor"],
)
fig.update_xaxes(
    showgrid=PLOT_CONFIG["showgrid_x"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(
    showgrid=PLOT_CONFIG["showgrid_y"],
    gridcolor=PLOT_CONFIG["gridcolor"],
    gridwidth=PLOT_CONFIG["gridwidth"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(tickformat=",.0f", row=1, col=1)
fig.update_yaxes(tickprefix="$", tickformat=".1f", row=1, col=2)

fig.show()

SSI OUTCOMES BY IMPUTATION MODEL
(Imputed net worth used as ssi_countable_resources)
Model                       Recipients  Expend ($B)  Avg Mo ($)   Recip %
--------------------------------------------------------------------------------
OLS                         11,342,398       100.46         738   153.3%
QuantReg                     8,262,234        74.58         752   111.7%
Matching                    11,016,779        89.95         680   148.9%
MDN                          9,778,097        86.88         740   132.1%
QRF                         12,083,642       102.83         709   163.3%
Baseline (no wealth)        19,769,600       152.59         643   267.2%
--------------------------------------------------------------------------------
Admin total                  7,400,000         59.6         697    100.0%


## Policy Reform: SSI Savings Penalty Elimination Act

The SSI resource limits ($2,000 individual / $3,000 couple) have been frozen since 1989. Adjusted for inflation, these thresholds are worth roughly $4,800 / $7,200 in 2024 dollars — meaning SSI's resource test has become substantially more restrictive over time without any legislative change.

The **SSI Savings Penalty Elimination Act** (S. 1234 / H.R. 2540, introduced April 2025) is a bipartisan bill that would raise the resource limits to $10,000 for individuals and $20,000 for couples, with future CPI indexing. This reform provides a natural test case for wealth imputation: simulating the effect of raising resource limits *requires* household-level wealth data, since the reform changes which households pass the resource test.

Below, we simulate SSI outcomes under both current law and the reformed thresholds for each imputation method. The difference — additional recipients and expenditure — represents the estimated reform impact. We also simulate the reform under the baseline (no wealth) scenario, where resources default to $0. Because every household already passes the resource test under current law in the baseline, raising the thresholds has *zero effect* — demonstrating that this reform literally cannot be simulated without imputed wealth data.

In [ ]:
# SSI Savings Penalty Elimination Act: Reform simulation

from policyengine_us import Microsimulation
from policyengine_us.model_api import Variable, YEAR, USD, where
from policyengine_us.entities import Person
from policyengine_core.reforms import Reform
from policyengine_us_data import EnhancedCPS_2024

# Define SPEA parameter reform: raise resource limits
spea_param_reform = Reform.from_dict(
    {
        "gov.ssa.ssi.eligibility.resources.limit.individual": {
            "2024-01-01.2100-12-31": 10_000,
        },
        "gov.ssa.ssi.eligibility.resources.limit.couple": {
            "2024-01-01.2100-12-31": 20_000,
        },
    },
    country_id="us",
)

# Reform simulation for each imputation method
ssi_reform_results = {}

for model_name, imputations in model_results.items():
    # Map household-level imputed net worth to person-level
    person_df = pd.DataFrame({"household_id": person_hh_ids})
    hh_df = pd.DataFrame(
        {
            "household_id": hh_ids,
            "networth": imputations["networth"].values.flatten(),
        }
    )
    merged = person_df.merge(hh_df, on="household_id", how="left")
    person_networth = merged["networth"].fillna(0).values

    # Assign to household heads only
    person_resources = np.where(is_head, np.maximum(person_networth, 0), 0.0).astype(
        np.float32
    )

    # Create resource override reform
    resource_reform = make_ssi_resource_reform(person_resources)

    # Compose the resource override + SPEA parameter reform as a tuple
    sim_reform = Microsimulation(
        dataset=EnhancedCPS_2024,
        reform=(resource_reform, spea_param_reform),
    )
    ssi_reform = sim_reform.calculate("ssi", period=2024)

    total_exp_reform = float(ssi_reform.sum())
    total_recip_reform = float((ssi_reform > 0).astype(float).sum())
    avg_mo_reform = (
        total_exp_reform / total_recip_reform / 12 if total_recip_reform > 0 else 0
    )

    ssi_reform_results[model_name] = {
        "recipients": total_recip_reform,
        "expenditure": total_exp_reform,
        "avg_monthly": avg_mo_reform,
    }

# Baseline reform (no wealth data)
# Under baseline, all resources = $0, so everyone already passes
# the $2K test. Raising it to $10K/$20K changes nothing.
sim_baseline_reform = Microsimulation(
    dataset=EnhancedCPS_2024,
    reform=spea_param_reform,
)
ssi_baseline_reform = sim_baseline_reform.calculate("ssi", period=2024)
baseline_reform_exp = float(ssi_baseline_reform.sum())
baseline_reform_recip = float((ssi_baseline_reform > 0).astype(float).sum())
baseline_reform_avg = (
    baseline_reform_exp / baseline_reform_recip / 12 if baseline_reform_recip > 0 else 0
)

ssi_reform_results["Baseline (no wealth)"] = {
    "recipients": baseline_reform_recip,
    "expenditure": baseline_reform_exp,
    "avg_monthly": baseline_reform_avg,
}

print("Reform simulation complete.")
print(f"Models simulated: {list(ssi_reform_results.keys())}")

Reform simulation complete.
Models simulated: ['OLS', 'QuantReg', 'Matching', 'MDN', 'QRF', 'Baseline (no wealth)']


In [ ]:
# Compute reform impact: difference between reform and current law 

reform_impact = {}
for model_name in ssi_reform_results:
    current = ssi_results[model_name]
    reformed = ssi_reform_results[model_name]
    reform_impact[model_name] = {
        "current_recipients": current["recipients"],
        "reform_recipients": reformed["recipients"],
        "additional_recipients": (reformed["recipients"] - current["recipients"]),
        "current_expenditure": current["expenditure"],
        "reform_expenditure": reformed["expenditure"],
        "additional_expenditure": (reformed["expenditure"] - current["expenditure"]),
        "pct_increase_recipients": (
            (reformed["recipients"] - current["recipients"])
            / current["recipients"]
            * 100
            if current["recipients"] > 0
            else 0
        ),
        "pct_increase_expenditure": (
            (reformed["expenditure"] - current["expenditure"])
            / current["expenditure"]
            * 100
            if current["expenditure"] > 0
            else 0
        ),
    }

# Display reform impact table
print("=" * 100)
print("SSI SAVINGS PENALTY ELIMINATION ACT: REFORM IMPACT BY IMPUTATION METHOD")
print("Resource limits: $2K/$3K (current) -> $10K/$20K (reform)")
print("=" * 100)
print(
    f"{'Model':<25} "
    f"{'Current':>10} "
    f"{'Reform':>10} "
    f"{'Add. Recip':>12} "
    f"{'Add. Exp($B)':>13} "
    f"{'% Incr':>8}"
)
print("-" * 100)
for model, r in reform_impact.items():
    print(
        f"{model:<25} "
        f"{r['current_recipients'] / 1e6:>10.2f}M "
        f"{r['reform_recipients'] / 1e6:>10.2f}M "
        f"{r['additional_recipients'] / 1e6:>12.2f}M "
        f"{r['additional_expenditure'] / 1e9:>13.2f} "
        f"{r['pct_increase_recipients']:>7.1f}%"
    )
print("-" * 100)
print(
    "\nBaseline (no wealth) reform impact = 0: "
    "without wealth data, this reform is cannot be simulated."
)

SSI SAVINGS PENALTY ELIMINATION ACT: REFORM IMPACT BY IMPUTATION METHOD
Resource limits: $2K/$3K (current) -> $10K/$20K (reform)
Model                        Current     Reform   Add. Recip  Add. Exp($B)   % Incr
----------------------------------------------------------------------------------------------------
OLS                            11.34M      15.57M         4.22M         24.10    37.2%
QuantReg                        8.26M       8.37M         0.11M          0.99     1.3%
Matching                       11.02M      12.86M         1.85M         10.41    16.8%
MDN                             9.78M      10.91M         1.13M          9.09    11.6%
QRF                            12.08M      12.71M         0.63M          3.39     5.2%
Baseline (no wealth)           19.77M      19.77M         0.00M          0.00     0.0%
----------------------------------------------------------------------------------------------------

Baseline (no wealth) reform impact = 0: without wealth data, t

In [ ]:
# Reform impact visualization
from pathlib import Path

FIGURES_DIR = Path(
    "/Users/movil1/Desktop/PYTHONJOBS/PolicyEngine/microimpute/paper/figures"
)

model_colors = {
    "OLS": "#CC6677",
    "QuantReg": "#DDCC77",
    "Matching": "#117733",
    "MDN": "#332288",
    "QRF": "#88CCEE",
    "Baseline (no wealth)": "#999999",
}

# Exclude baseline from the main comparison chart since its
# reform impact is 0 and distorts the scale
plot_models = [m for m in reform_impact if m != "Baseline (no wealth)"]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "SSI recipients: current law vs reform",
        "SSI expenditure: current law vs reform ($B)",
    ],
    horizontal_spacing=0.12,
)

# Current law bars
fig.add_trace(
    go.Bar(
        x=plot_models,
        y=[ssi_results[m]["recipients"] for m in plot_models],
        name="Current law",
        marker_color=[model_colors.get(m, "#999999") for m in plot_models],
        opacity=0.5,
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Reform bars
fig.add_trace(
    go.Bar(
        x=plot_models,
        y=[ssi_reform_results[m]["recipients"] for m in plot_models],
        name="Reform ($10K/$20K)",
        marker_color=[model_colors.get(m, "#999999") for m in plot_models],
        opacity=1.0,
        showlegend=True,
    ),
    row=1,
    col=1,
)

# Admin total reference line
fig.add_hline(
    y=admin_recipients,
    line_dash="dash",
    line_color="black",
    line_width=1,
    row=1,
    col=1,
)

# Expenditure - current law
fig.add_trace(
    go.Bar(
        x=plot_models,
        y=[ssi_results[m]["expenditure"] / 1e9 for m in plot_models],
        name="Current law",
        marker_color=[model_colors.get(m, "#999999") for m in plot_models],
        opacity=0.5,
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Expenditure - reform
fig.add_trace(
    go.Bar(
        x=plot_models,
        y=[ssi_reform_results[m]["expenditure"] / 1e9 for m in plot_models],
        name="Reform ($10K/$20K)",
        marker_color=[model_colors.get(m, "#999999") for m in plot_models],
        opacity=1.0,
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.add_hline(
    y=admin_expenditure / 1e9,
    line_dash="dash",
    line_color="black",
    line_width=1,
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(dash="dash", color="black", width=1),
        name="Admin total (current law)",
        showlegend=True,
    ),
    row=1,
    col=1,
)

fig.update_layout(
    title=("SSI Savings Penalty Elimination Act: Reform impact by imputation method"),
    width=PLOT_CONFIG["width"] + 200,
    height=PLOT_CONFIG["height"] - 100,
    paper_bgcolor=PLOT_CONFIG["paper_bgcolor"],
    plot_bgcolor=PLOT_CONFIG["plot_bgcolor"],
    barmode="group",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=1.02,
    ),
)
fig.update_xaxes(
    showgrid=PLOT_CONFIG["showgrid_x"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(
    showgrid=PLOT_CONFIG["showgrid_y"],
    gridcolor=PLOT_CONFIG["gridcolor"],
    gridwidth=PLOT_CONFIG["gridwidth"],
    showline=PLOT_CONFIG["showline"],
    linecolor=PLOT_CONFIG["linecolor"],
)
fig.update_yaxes(tickformat=",.0f", row=1, col=1)
fig.update_yaxes(tickprefix="$", tickformat=".1f", row=1, col=2)

# Save figure for paper
output_path = FIGURES_DIR / "models_ssi_reform_comparison.png"
fig.write_image(
    str(output_path),
    width=1200,
    height=500,
    scale=2,
)
fig.show()